In [1]:
!pip uninstall -y torchao unsloth unsloth_zoo -q

In [2]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 722.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 18.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 25.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 17.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 13.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 7.7 M

In [4]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit", # Llama 3.2 vision support
    "unsloth/Llama-3.2-11B-Vision-bnb-4bit",
    "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit", # Can fit in a 80GB card!
    "unsloth/Llama-3.2-90B-Vision-bnb-4bit",

    "unsloth/Pixtral-12B-2409-bnb-4bit",              # Pixtral fits in 16GB!
    "unsloth/Pixtral-12B-Base-2409-bnb-4bit",         # Pixtral base model

    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",          # Qwen2 VL support
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    "unsloth/Qwen2-VL-72B-Instruct-bnb-4bit",

    "unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit",      # Any Llava variant works!
    "unsloth/llava-1.5-7b-hf-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    #"Arup330/Neck_cot_llama_lora",
    "unsloth/Llama-3.2-11B-Vision-Instruct",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Mllama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 7.231 GiB
no_split classes   : ['MllamaCrossAttentionDecoderLayer', 'MllamaSelfAttentionDecoderLayer', 'MllamaVisionEncoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.342 GiB
activation reserve : 9.188 GiB requested
  cuda:0  budget  12.95 GiB  weights  3.760 GiB  free  9.192 GiB  reserve  9.188 GiB
  cuda:1  budget  13.00 GiB  weights  3.471 GiB  free  9.525 Gi

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

In [5]:
# ---------------------------------------------------------------------------
# 2. Attach LoRA adapters
# ---------------------------------------------------------------------------
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)


In [6]:
 
# ---------------------------------------------------------------------------
# 3. Load your CoT-augmented train CSVs (output of the CoT generation step)
# ---------------------------------------------------------------------------
import os
import glob
import pandas as pd
from PIL import Image
 
#COT_DATA_ROOT = "/kaggle/input/datasets/arups330/slackdataset/Neck_train_with_CoT"   # <-- output folder from your CoT step
#COT_DATA_ROOT = "/kaggle/input/datasets/arups330/slackdataset/BrainFaceT_train_with_CoT" 
#csv_paths = glob.glob(os.path.join(COT_DATA_ROOT, "*", "*", "*.csv"))  # CT/train/*.csv, MRI/train/*.csv
COT_DATA_ROOT = "/kaggle/input/datasets/arups330/slackdataset/Abdomen_Train_with_CoT/CT/train"
csv_paths  =  [
    os.path.join(COT_DATA_ROOT, "closed_with_CoT.csv"),
    os.path.join(COT_DATA_ROOT, "open_with_CoT.csv"),
]
print(f"Found {len(csv_paths)} CoT CSVs:")
for p in csv_paths:
    print(" ", p)
 
frames = []
for csv_path in csv_paths:
    split_dir = os.path.dirname(csv_path)
    df = pd.read_csv(csv_path)
    df["split_dir"] = split_dir
    frames.append(df)
 
cot_df = pd.concat(frames, ignore_index=True)
 
# Drop any rows where CoT generation failed (saved as empty string earlier)
before = len(cot_df)
cot_df = cot_df[cot_df["CoT"].notna() & (cot_df["CoT"].str.strip() != "")]
print(f"Loaded {before} rows, {len(cot_df)} have valid CoT (dropped {before - len(cot_df)} empty/failed rows)")
 
IMG_COL = "image_file" if "image_file" in cot_df.columns else "img_name"
 
def resolve_image_path(split_dir: str, img_name: str) -> str:
    flat_name = os.path.basename(str(img_name))
    candidates = [
        os.path.join(split_dir, str(img_name)),
        os.path.join(split_dir, flat_name),
        os.path.join(split_dir, str(img_name).replace("/", "_")),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Could not find image for {img_name!r} in {split_dir}")


Found 2 CoT CSVs:
  /kaggle/input/datasets/arups330/slackdataset/Abdomen_Train_with_CoT/CT/train/closed_with_CoT.csv
  /kaggle/input/datasets/arups330/slackdataset/Abdomen_Train_with_CoT/CT/train/open_with_CoT.csv
Loaded 300 rows, 300 have valid CoT (dropped 0 empty/failed rows)


In [7]:
# ---------------------------------------------------------------------------
# 4. convert_to_conversation -- same style as Unsloth's radiography example,
#    but: instruction includes the question + ground-truth answer, and the
#    assistant target is the generated CoT (not a short caption).
# ---------------------------------------------------------------------------
instruction_template = (
    "You are an expert radiologist. Analyze this medical image and provide "
    "detailed step-by-step clinical reasoning.\n\n"
    "Question: {question}\n"
    "Ground Truth Answer: {answer}\n\n"
    "Provide your full diagnostic reasoning chain, covering imaging modality, "
    "anatomical region, region-wise findings, key visual features, any "
    "abnormality detected, clinical interpretation, and justification for "
    "the answer above."
)
 
def convert_to_conversation(sample):
    instruction = instruction_template.format(
        question=sample["question"], answer=sample["answer"]
    )
    image_path = resolve_image_path(sample["split_dir"], sample[IMG_COL])
    image = Image.open(image_path).convert("RGB")
 
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": image},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample["CoT"]}],
        },
    ]
    return {"messages": conversation}
pass
 
print("Converting rows to Unsloth chat format...")
converted_dataset = [convert_to_conversation(row) for _, row in cot_df.iterrows()]
print(f"Done. {len(converted_dataset)} examples ready.")
print("Example:", converted_dataset[0]["messages"])


Converting rows to Unsloth chat format...
Done. 300 examples ready.
Example: [{'role': 'user', 'content': [{'type': 'text', 'text': 'You are an expert radiologist. Analyze this medical image and provide detailed step-by-step clinical reasoning.\n\nQuestion: Does the picture contain liver?\nGround Truth Answer: No\n\nProvide your full diagnostic reasoning chain, covering imaging modality, anatomical region, region-wise findings, key visual features, any abnormality detected, clinical interpretation, and justification for the answer above.'}, {'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=512x512 at 0x7A3D11799E50>}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': '**Step 1: Imaging Modality**\n\nThe provided image appears to be a computed tomography (CT) scan. This is evident from the cross-sectional view of the body, the use of grayscale to represent different tissue densities, and the presence of a clear outline of the body\'s internal structures. The ima

In [8]:
# ---------------------------------------------------------------------------
# 5. Quick check BEFORE fine-tuning (same as the Unsloth notebook does --
#    run one inference with the base model to see what it currently outputs)
# ---------------------------------------------------------------------------
FastVisionModel.for_inference(model)
sample = cot_df.iloc[0]
test_instruction = instruction_template.format(question=sample["question"], answer=sample["answer"])
test_image = Image.open(resolve_image_path(sample["split_dir"], sample[IMG_COL])).convert("RGB")
 
messages = [{"role": "user", "content": [
    {"type": "image"}, {"type": "text", "text": test_instruction}
]}]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(test_image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
output_ids = model.generate(**inputs, max_new_tokens=256, use_cache=True, temperature=1.5, min_p=0.1)
print("\nBEFORE fine-tuning, model output:")
print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))


[unsloth_zoo.log|WARNING]Unsloth: torch.compile hit one of Unsloth's own `torch.compiler.disable`d gradient-checkpointing hooks inside MllamaPrecomputedAspectRatioEmbedding_forward; running it eagerly from here. Training is unaffected apart from speed.
[unsloth_zoo.log|WARNING]Unsloth: torch.compile hit one of Unsloth's own `torch.compiler.disable`d gradient-checkpointing hooks inside MllamaPrecomputedPositionEmbedding_forward; running it eagerly from here. Training is unaffected apart from speed.



BEFORE fine-tuning, model output:
The image is a chest computed tomography (CT) scan, which provides detailed cross-sectional images of the lungs, surrounding tissues, and organs. 

**Step 1: Identifying Anatomical Region**
The provided image appears to be focused on the chest, including the thorax, lungs, and surrounding structures, excluding any visible liver tissue.

**Step 2: Visual Inspection**
Upon visual inspection, the image shows a clear representation of the thoracic region, with the lungs taking up significant space within the thorax. The surrounding structures, including the trachea, esophagus, and diaphragm, are also visible. The lungs appear to be normally aerated, with visible aeration and bronchovascular structures evident.

**Step 3: Key Features**
Key features of the lungs include a clear division between the right and left lung lobes and the presence of the mediastinal structures in between, which are all consistent with the normal anatomical arrangement.

**Step 4:

In [9]:
# ---------------------------------------------------------------------------
# 6. Train with SFTTrainer + UnslothVisionDataCollator
# ---------------------------------------------------------------------------
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
 
FastVisionModel.for_training(model)
 
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=converted_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,   # CoT targets are long -- keep batch size low
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=30,                  # quick test run -- comment out for full training
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
 
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        dataset_num_proc=4,
        max_seq_length=4096,   # CoT outputs are long (9 sections) -- increased from 2048
    ),
)
 


Unsloth: You set `max_seq_length` as 4096 but the maximum the model supports is 2048. We shall reduce it.


In [10]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'bos_token_id': 128000}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 300 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 67,174,400 of 10,737,395,235 (0.63% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.192061
2,1.315013
3,1.296215
4,1.100355
5,0.912181
6,0.864945
7,0.804960
8,0.695640
9,0.569549
10,0.494294


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-30/tokenizer_config.json.


In [11]:
# ---------------------------------------------------------------------------
# 7. Check AFTER fine-tuning -- same sample as before, compare outputs
# ---------------------------------------------------------------------------
FastVisionModel.for_inference(model)
output_ids = model.generate(**inputs, max_new_tokens=1024, use_cache=True, temperature=0.3, min_p=0.1)
print("\nAFTER fine-tuning, model output:")
print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
print("\n(Compare this to the actual generated CoT it was trained on:)")
print(sample["CoT"])
 



AFTER fine-tuning, model output:
**Step 1: Imaging Modality**

The provided image appears to be a computed tomography (CT) scan, as indicated by the cross-sectional view of the body and the use of X-ray technology to create detailed internal images. The image has a grayscale pattern with varying densities, which is characteristic of CT scans.

**Step 2: Global Image Understanding**

The image shows a cross-sectional view of the chest, with the lungs, heart, and surrounding structures visible. The image is oriented in a way that the top of the image corresponds to the head, and the bottom corresponds to the feet.

**Step 3: Region-wise Analysis**

The image can be divided into several regions, including the lungs, heart, and mediastinum. Each region can be analyzed separately to identify any abnormalities.

**Step 4: Visual Feature Extraction**

Upon examining the image, there are no visible abnormalities in the lungs, heart, or surrounding structures. The lungs appear to be of normal 

In [12]:
# ---------------------------------------------------------------------------
# 8. Save the LoRA adapter locally, and optionally push to Hugging Face Hub
# ---------------------------------------------------------------------------
# model.save_pretrained("Neck_cot_llama_lora")
# tokenizer.save_pretrained("Neck_cot_llama_lora")
# print("\nSaved locally to ./Neck_cot_llama_lora")

model.save_pretrained("Abdomen_CoT_llama_lora")
tokenizer.save_pretrained("Abdomen_CoT_llama_lora")
print("\nSaved locally to ./Abdomen_CoT_llama_lora")


Unsloth: Restored added_tokens_decoder metadata in Abdomen_CoT_llama_lora/tokenizer_config.json.



Saved locally to ./Abdomen_CoT_llama_lora


In [13]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
# model.push_to_hub("Arup330/Neck_cot_llama_lora", token=HF_TOKEN)
# tokenizer.push_to_hub("Arup330/Neck_cot_llama_lora", token=HF_TOKEN)
#model.push_to_hub("Arup330/BrainFaceWithNeck_CoT_llama_lora", token=HF_TOKEN)
#tokenizer.push_to_hub("Arup330/BrainFaceWithNeck_CoT_llama_lora", token=HF_TOKEN)
model.push_to_hub("Arup330/Abdomen_CoT_llama_lora", token=HF_TOKEN)
tokenizer.push_to_hub("Arup330/Abdomen_CoT_llama_lora", token=HF_TOKEN)


README.md:   0%|          | 0.00/599 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Arup330/Abdomen_CoT_llama_lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp58lpkulb/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            